<a href="https://colab.research.google.com/github/azkakhan4/flyrank-ai-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/azkakhan4/flyrank-ai-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My proposed rule

I will prioritize content that has not been updated recently but still receives search impressions. Such content may be a good candidate for a refresh review.

I will examine two signals:
1. **Content staleness:** How long it has been since the content was last optimized. This is linked to FlyRank's existing content-refresh flags.
2. **Search impressions:** How often content appears in search results. This helps identify content that still has search visibility.

**Reason code:** `STALE_WITH_VISIBILITY`

**Action label:** `REVIEW_FOR_REFRESH`

The rule will use these signals to rank potential content-refresh opportunities. It will support human review, not automatically decide which content must be refreshed.

In [2]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/azkakhan4/flyrank-ai-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Signal 1: Content staleness
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
).astype("object").fillna("unknown")

staleness_table = df.groupby(
    "staleness_bucket", observed=False
).agg(
    n=("content_id", "size"),
    median_impressions=("impressions_90d", "median"),
    pages_with_500_plus_impressions=(
        "impressions_90d", lambda x: (x >= 500).sum()
    )
)

print("SIGNAL 1: CONTENT STALENESS")
display(staleness_table)

# Signal 2: Search impressions
df["impression_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 100, 500, 3000, 30000, np.inf],
    labels=["1-99", "100-499", "500-2999", "3000-29999", "30000+"],
    right=False
)

impressions_table = df.groupby(
    "impression_bucket", observed=False
).agg(
    n=("content_id", "size"),
    median_days_since_update=("days_since_last_update", "median"),
    pages_stale_180_plus=(
        "days_since_last_update", lambda x: (x >= 180).sum()
    )
)

print("\nSIGNAL 2: SEARCH IMPRESSIONS")
display(impressions_table)

SIGNAL 1: CONTENT STALENESS


,n,median_impressions,pages_with_500_plus_impressions
staleness_bucket,,,
0-30,20480,470.0,10063
181-365,169,16.0,17
31-90,175,510.0,88
365+,5,2.0,0
91-180,9171,1692.0,6558



SIGNAL 2: SEARCH IMPRESSIONS


,n,median_days_since_update,pages_stale_180_plus
impression_bucket,,,
1-99,7994,20.0,139
100-499,5280,22.0,18
500-2999,8443,22.0,8
3000-29999,7205,22.0,7
30000+,1078,25.0,2


In [3]:
print(df["days_since_last_update"].describe())

print(
    df.nlargest(5, "days_since_last_update")[
        ["content_id", "days_since_last_update", "impressions_90d"]
    ]
)

count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64
                 content_id  days_since_last_update  impressions_90d
4606   content_3f3576c295f5                     373                1
26242  content_55a5b1c46474                     373               35
29384  content_f6fdf87348f6                     373                2
18440  content_8d56efff1e71                     372                1
24216  content_1b4ec72dafd4                     372                2


### Signal findings and verdicts

**Signal 1: Content staleness — MIXED**

Content aged 91–180 days has a median of 1,692 impressions. However, content aged 181–365 days has only 16 median impressions, and the 365+ bucket has only 2. The oldest buckets have very small sample sizes, so their results should be treated cautiously.

**Signal 2: Search impressions — OPPOSITE**

Higher-impression buckets generally contain fewer pages that have been stale for 180 days or more. This does not support the assumption that high-impression content is generally stale.

**Decision:** Staleness may help identify refresh opportunities, but the rule must account for mixed evidence and avoid treating all old content as a good opportunity.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# Build the baseline action score

queue = df.copy()

# Score: prioritize stale content that still has search visibility
queue["action_score"] = (
    (queue["days_since_last_update"] >= 90).astype(int) * 2
    + (queue["impressions_90d"] >= 500).astype(int)
)

# One reason code
queue["reason_code"] = np.where(
    queue["action_score"] > 0,
    "STALE_WITH_VISIBILITY",
    "LOW_REFRESH_PRIORITY"
)

# Action label
queue["action_label"] = np.where(
    queue["action_score"] > 0,
    "REVIEW_FOR_REFRESH",
    "NO_ACTION"
)

# Rank the queue
queue = queue.sort_values(
    "action_score", ascending=False
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

# Save the ranked queue
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Ranked queue created successfully!")
print("Total content items:", len(queue))
print("Saved to:", output_path)

display(queue.head(10))

Ranked queue created successfully!
Total content items: 30000
Saved to: work/outputs/baseline_action_score.csv


,rank,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,staleness_bucket,impression_bucket,action_score,reason_code,action_label
0,1,content_3415015690ae,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6073.0,...,0.00,good,striking,stable,-6.1,91-180,3000-29999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
1,2,content_0138556d5d08,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,commercial,5268.0,...,2.08,good,striking,up,346.2,91-180,3000-29999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
2,3,content_018cf0021b5d,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,4802.0,...,0.00,moderate,page_1,down,-53.0,91-180,500-2999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
3,4,content_9dedbc330e1f,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,7877.0,...,0.00,moderate,page_3_5,down,-40.1,91-180,500-2999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
4,5,content_7638b4f1674a,client_4e07408562,10.0,1.00,HIGH,0.13,keyword article,transactional,1462.0,...,0.00,good,page_1,up,22.1,91-180,3000-29999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
5,6,content_ea07246641f4,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,1522.0,...,0.00,good,page_1,down,-56.9,91-180,3000-29999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
6,7,content_20decd85a0c2,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,5679.0,...,0.00,good,page_3_5,down,-75.2,91-180,3000-29999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
7,8,content_3b806fcc5b2c,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6289.0,...,0.00,good,page_1,down,-52.3,91-180,3000-29999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
8,9,content_73cf70f08e06,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,transactional,NaN,...,0.00,good,top_3,down,-49.9,91-180,3000-29999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
9,10,content_77867ed726e1,client_19581e27de,30.0,0.29,LOW,0.08,keyword article,transactional,NaN,...,0.00,good,striking,down,-28.5,91-180,3000-29999,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top_10 = queue.head(10).copy()

review_columns = [
    "rank",
    "content_id",
    "days_since_last_update",
    "impressions_90d",
    "action_score",
    "reason_code",
    "action_label"
]

display(top_10[review_columns])

,rank,content_id,days_since_last_update,impressions_90d,action_score,reason_code,action_label
0,1,content_3415015690ae,104,5237,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
1,2,content_0138556d5d08,104,5929,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
2,3,content_018cf0021b5d,104,1174,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
3,4,content_9dedbc330e1f,104,1772,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
4,5,content_7638b4f1674a,104,4122,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
5,6,content_ea07246641f4,104,3535,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
6,7,content_20decd85a0c2,104,5061,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
7,8,content_3b806fcc5b2c,104,3085,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
8,9,content_73cf70f08e06,104,28950,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH
9,10,content_77867ed726e1,104,6515,3,STALE_WITH_VISIBILITY,REVIEW_FOR_REFRESH


### Top-10 review

1. Rank 1 — Review for refresh. It is 104 days old and has search visibility. This could be wrong if the content is no longer relevant or its impressions are declining.
2. Rank 2 — Review for refresh. It is 104 days old and has 5,929 impressions. This could be wrong if the page is already performing well and needs no changes.
3. Rank 3 — Review for refresh. It is 104 days old and has 1,174 impressions. This could be wrong if the impressions are not bringing useful visitors.
4. Rank 4 — Review for refresh. It is 104 days old and has 1,772 impressions. This could be wrong if the content is still accurate and useful.
5. Rank 5 — Review for refresh. It is 104 days old and has 4,122 impressions. This could be wrong if the page does not need updating.
6. Rank 6 — Review for refresh. It is 104 days old and has 3,535 impressions. This could be wrong if its search performance is already satisfactory.
7. Rank 7 — Review for refresh. It is 104 days old and has 5,061 impressions. This could be wrong if the content is still relevant and up to date.
8. Rank 8 — Review for refresh. It is 104 days old and has 3,085 impressions. This could be wrong if the page has no meaningful content gaps.
9. Rank 9 — Review for refresh. It is 104 days old and has 28,950 impressions. This could be wrong if the page is already performing strongly.
10. Rank 10 — Review for refresh. It is 104 days old and has 6,515 impressions. This could be wrong if a refresh would not improve its usefulness.

**Confidence note:** These are initial review recommendations based on staleness and impressions only. A human should check the content before deciding to refresh it.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Check the weakest recommendations in the ranked queue

weak_picks = queue.tail(5)[
    [
        "rank",
        "days_since_last_update",
        "impressions_90d",
        "action_score",
        "action_label"
    ]
]

display(weak_picks)

,rank,days_since_last_update,impressions_90d,action_score,action_label
29995,29996,20,90,0,NO_ACTION
29996,29997,20,1,0,NO_ACTION
29997,29998,20,1,0,NO_ACTION
29998,29999,20,349,0,NO_ACTION
29999,30000,22,410,0,NO_ACTION


### Weak picks and leakage check

**Weak picks:** The lowest-ranked records have very low action scores, indicating low refresh priority under the baseline rule. Some records may still be highly stale but receive a low score because they have low impressions. This shows that the rule does not capture every potentially useful refresh opportunity.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.